# STAC data prep — Soil sand content, one COG per depth (2D)

Notebook to migrate the six depth intervals of the SoilGrids 2.0 sand content (5 km, from Google Earth Engine) to **Cloud Optimized GeoTIFFs (CoG)**.
Same pattern as `20_groundwater_recharge.ipynb`: one collection with one item (COG) per depth interval, published in GeoServer WMS.

This is the **COG variant** of the dataset. The **Zarr variant** (one `(depth, y, x)` cube) is `30_soil_sand_zarr.ipynb`; run both to compare them.

- **Case:** 2D rasters, continuous (%), Africa and surroundings, 2076 x 1671 pixels (about 0.045 degree), 14-15 MB each.
- **Format:** one COG per depth (no tiling needed for this size, see `DATA_FORMATS.md`); one STAC item per depth so each depth is a separate WMS layer.
- **Resampling:** `AVERAGE` for overviews, because the values are continuous.
- **Precision:** the sources are `float64`, the COGs `float32` (about 7 significant digits, far more than the model accuracy). The check compares against the `float32` cast of the source.
- **Nodata:** the sources have no nodata value, empty cells are `NaN` (about 53 % of the grid). The COGs use `-9999`.
- **Metadata template:** `metadata/metadata_Soil_Sand_Content.json`. The empty fields are filled from the rasters below.

Sources (`P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian`): `sand_5km_<depth>.tif` for the depths 0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100-200cm.

**Running in docker:** the `P:` drive is not visible in the container, so run from the host:

```powershell
docker\run_notebook_with_p_drive.ps1 -Notebook 28_soil_sand_cog.ipynb -Variable soil_sand_cog -InputPattern "sand_5km_*.tif"
```

### Import packages

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import rasterio
import rioxarray as rxr

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
COLLECTION_ID = "soil_sand_cog"
PROP = "sand"
DEPTHS = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm', '100-200cm']

# One entry per source file: the item id equals the file name (without .tif)
VARIANTS = {}
for depth in DEPTHS:
    top, bottom = (int(v) for v in depth.removesuffix("cm").split("-"))
    VARIANTS[f"{PROP}_5km_{depth}"] = {
        "DEPTH_RANGE": f"{top}-{bottom} cm",
        "DEPTH_TOP_CM": top,
        "DEPTH_BOTTOM_CM": bottom,
        "TITLE": f"Sand content, {top}-{bottom} cm depth (5 km)",
        "DESCRIPTION": f"Sand content (mean of the {top}-{bottom} cm depth interval, SoilGrids 2.0) at about 5 km resolution, in %.",
    }
OUT_NODATA = -9999.0
UNITS = "%"
# The maps are static (no time series). SoilGrids 2.0 was released in 2020, which is used as the validity period.
TEMPORAL_EXTENT = ['2020-01-01T00:00:00Z', '2020-12-31T23:59:59Z']
# The template says 'mass fraction in permille', but the 5 km maps from Google Earth Engine are in % (clay 0-68, sand 11-100)
DESCRIPTION_FIX = ("mass fraction in \u2030", "mass fraction in % (g/100 g)")

# Same layout as P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian (DATA_VIVIAN is set in docker,
# where the P: drive is not mounted; a local mirror of that folder is used and the stac/ output is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_tifs = {name: data_vivian / f"{name}.tif" for name in VARIANTS}
metadata_template_path = stac_repo / "metadata" / "metadata_Soil_Sand_Content.json"
if not metadata_template_path.is_file():  # same file as in the P: metadata folder
    metadata_template_path = data_vivian / "metadata" / "metadata_Soil_Sand_Content.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
cog_dir = processed_data_dir / "cog"
cog_dir.mkdir(parents=True, exist_ok=True)

for p in (*source_tifs.values(), metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Sources:", *source_tifs.values(), sep="\n  ")
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Sources:
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_0-5cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_5-15cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_15-30cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_30-60cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_60-100cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_100-200cm.tif
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_Soil_Sand_Content.json
Output dir: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog


### Read raw data

In [3]:
# mask_and_scale=False keeps the stored values and the original nodata
raw = {}
for name, path in source_tifs.items():
    raw[name] = rxr.open_rasterio(path, mask_and_scale=False)
    with rasterio.open(path) as src:
        print(f"{name}: {src.width} x {src.height} | dtype {src.dtypes[0]} | nodata {src.nodata} | crs {src.crs.to_epsg()}")
        print(f"   bounds {tuple(round(v, 4) for v in src.bounds)} | res {tuple(round(v, 6) for v in src.res)}")
        print(f"   tiled {src.is_tiled} | compression {src.compression} | overviews {src.overviews(1)}")

# All depths must be on the same grid to be published as one collection
ref_name, *other_names = list(raw)
for other in other_names:
    assert raw[ref_name].shape == raw[other].shape, "Different grid shapes"
    assert np.allclose(raw[ref_name].rio.bounds(), raw[other].rio.bounds()), "Different bounds"
raw[ref_name]

sand_5km_0-5cm: 1671 x 2076 | dtype float64 | nodata None | crs 4326
   bounds (-20.0324, -46.6226, 55.0218, 46.6226) | res (0.044916, 0.044916)
   tiled True | compression Compression.lzw | overviews []
sand_5km_5-15cm: 1671 x 2076 | dtype float64 | nodata None | crs 4326
   bounds (-20.0324, -46.6226, 55.0218, 46.6226) | res (0.044916, 0.044916)
   tiled True | compression Compression.lzw | overviews []
sand_5km_15-30cm: 1671 x 2076 | dtype float64 | nodata None | crs 4326
   bounds (-20.0324, -46.6226, 55.0218, 46.6226) | res (0.044916, 0.044916)
   tiled True | compression Compression.lzw | overviews []
sand_5km_30-60cm: 1671 x 2076 | dtype float64 | nodata None | crs 4326
   bounds (-20.0324, -46.6226, 55.0218, 46.6226) | res (0.044916, 0.044916)
   tiled True | compression Compression.lzw | overviews []
sand_5km_60-100cm: 1671 x 2076 | dtype float64 | nodata None | crs 4326
   bounds (-20.0324, -46.6226, 55.0218, 46.6226) | res (0.044916, 0.044916)
   tiled True | compression Com

<xarray.DataArray (band: 1, y: 2076, x: 1671)> Size: 28MB
[3468996 values with dtype=float64]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 17kB 46.6 46.56 46.51 46.47 ... -46.51 -46.56 -46.6
  * x            (x) float64 13kB -20.01 -19.97 -19.92 ... 54.91 54.95 55.0
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    scale_factor:   1.0
    add_offset:     0.0
    long_name:      b0

### Check CF compliancy original file

Not applicable for a plain GeoTIFF (same approach as `11b_Salinity` and `17_seabed_litho`): CRS, nodata and attributes are written explicitly below.

### Make CF compliant alterations (dataset dependent)

- Harmonise the nodata value to `-9999` (a pixel is nodata if it is not finite) and cast `float64` to `float32`.
- Fill the metadata fields that are empty in the template using the rasters: `SPATIAL_EXTENT`, `SPATIAL_RESOLUTION`, `NODATA`, `DATA_TYPE`, `DIMENSIONS`, `MEDIA_TYPE`, `DATA_MODEL`, `COLLECTION_ID`, `UNITS`, ...
- `ITEMS` holds the per-depth title and description for the STAC builder.

In [4]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

layers = {}
for name, ds in raw.items():
    da = ds.isel(band=0).drop_vars("band")
    src_nodata = ds.rio.nodata
    invalid = ~np.isfinite(da.values)
    if src_nodata is not None:
        invalid |= np.isclose(da.values, src_nodata)
    da = da.where(~invalid, OUT_NODATA).astype("float32")
    da = da.rio.write_crs(4326).rio.write_nodata(OUT_NODATA)
    da.attrs["_FillValue"] = OUT_NODATA
    da.name = name
    layers[name] = da
    valid = da.values[~invalid]
    print(f"{name}: nodata {src_nodata} -> {OUT_NODATA} (NaN) | valid pixels {valid.size} of {invalid.size} | "
          f"min {valid.min():.2f} mean {valid.mean():.2f} max {valid.max():.2f}")

# Bounds are pixel edges; the STAC bbox must stay inside the valid EPSG:4326 range
left, bottom, right, top = layers[ref_name].rio.bounds()
bbox = [max(left, -180.0), max(bottom, -90.0), min(right, 180.0), min(top, 90.0)]
res_x, res_y = layers[ref_name].rio.resolution()

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "SoilSand",
    "DESCRIPTION": metadata["DESCRIPTION"].replace(*DESCRIPTION_FIX),
    "HISTORY": [
        "SoilGrids 2.0 sand content (ISRIC), mean per depth interval, downloaded at 5 x 5 km from Google Earth Engine",
        "One file per standard depth interval: 0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100-200cm",
        "Converted to Cloud Optimized GeoTIFFs (float32, DEFLATE, floating point predictor, average overviews), nodata set to -9999",
    ],
    "MEDIA_TYPE": "image/tiff; application=geotiff; profile=cloud-optimized",
    "DATA_MODEL": "raster_cog",
    "DIMENSIONS": ["x", "y"],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    "TEMPORAL_EXTENT": TEMPORAL_EXTENT,
    "KEYWORDS": ['sand content', 'soil texture', 'soil', 'SoilGrids', 'depth intervals'] + ["cog"],
    "TAGS": ['soil', 'sand', 'texture', 'cog'],
    "LONG_NAME": "Sand content (mean per depth interval)",
    "UNITS": UNITS,
    "COMMENT": (
        "Six depth intervals are included as separate items: 0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100-200cm. Values are the mean sand content "
        "per depth interval in % (g/100 g), predicted by SoilGrids 2.0 and aggregated to a 5 km grid via Google Earth Engine. "
        f"{OUT_NODATA:.0f} is nodata (cells without prediction). The files are static maps, not a time series. "
        "Float64 values of the source are stored as float32."
    ),
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": round(abs(res_x), 4),
    "NODATA": OUT_NODATA,
    "DATA_TYPE": "float32",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": COLLECTION_ID,  # GeoServer workspace used by scripts/28_soil_sand_cog.ipynb
    "ITEMS": VARIANTS,
})

# Add attributes to the rasters (GeoTIFF tags only hold scalars)
for name, da in layers.items():
    for attr_name, attr_val in metadata.items():
        if attr_name == "ITEMS":
            continue
        if attr_name == "PROVIDERS":
            attr_val = json.dumps(attr_val)
        if attr_name == "MEDIA_TYPE":
            attr_val = "IMAGE/TIFF"  # same convention as 11b_Salinity
        if isinstance(attr_val, list):
            attr_val = json.dumps(attr_val)
        da.attrs[attr_name] = attr_val
    da.attrs["TITLE"] = VARIANTS[name]["TITLE"]
    da.attrs["DEPTH_RANGE"] = VARIANTS[name]["DEPTH_RANGE"]
    da.attrs["Conventions"] = "CF-1.8"

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("SPATIAL_RESOLUTION:", metadata["SPATIAL_RESOLUTION"])
print("NODATA:", metadata["NODATA"], "| DATA_TYPE:", metadata["DATA_TYPE"])

sand_5km_0-5cm: nodata None -> -9999.0 (NaN) | valid pixels 1617450 of 3468996 | min 12.22 mean 67.54 max 99.91
sand_5km_5-15cm: nodata None -> -9999.0 (NaN) | valid pixels 1617450 of 3468996 | min 12.33 mean 67.06 max 99.90
sand_5km_15-30cm: nodata None -> -9999.0 (NaN) | valid pixels 1617450 of 3468996 | min 12.36 mean 65.66 max 100.00
sand_5km_30-60cm: nodata None -> -9999.0 (NaN) | valid pixels 1617450 of 3468996 | min 11.88 mean 64.40 max 100.00
sand_5km_60-100cm: nodata None -> -9999.0 (NaN) | valid pixels 1617450 of 3468996 | min 11.66 mean 64.84 max 100.00
sand_5km_100-200cm: nodata None -> -9999.0 (NaN) | valid pixels 1617450 of 3468996 | min 11.32 mean 65.47 max 100.00
SPATIAL_EXTENT: [-20.032431, -46.622563, 55.021811, 46.622563]
SPATIAL_RESOLUTION: 0.0449
NODATA: -9999.0 | DATA_TYPE: float32


### Write data to CoG

One CoG per depth, `DEFLATE` compression with the floating point predictor, **average** overviews because the data are continuous.

In [5]:
out_paths = {}
for name, da in layers.items():
    out_paths[name] = cog_dir / f"{name}.tif"
    da.rio.to_raster(
        out_paths[name],
        driver="COG",
        compress="DEFLATE",
        predictor="FLOATING_POINT",
        dtype="float32",
        overview_resampling="AVERAGE",
        resampling="NEAREST",
    )
    print("Wrote COG:", out_paths[name], f"({out_paths[name].stat().st_size / 1e6:.2f} MB)")

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print("Wrote metadata:", metadata_output_path)

Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog\cog\sand_5km_0-5cm.tif (6.43 MB)
Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog\cog\sand_5km_5-15cm.tif (6.44 MB)
Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog\cog\sand_5km_15-30cm.tif (6.46 MB)
Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog\cog\sand_5km_30-60cm.tif (6.47 MB)
Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog\cog\sand_5km_60-100cm.tif (6.45 MB)
Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog\cog\sand_5km_100-200cm.tif (6.33 MB)
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_cog\metadata_soil_sand_cog.json


### Verify the output

In [6]:
for name, out_path in out_paths.items():
    with rasterio.open(out_path) as src:
        assert src.dtypes[0] == "float32"
        assert src.nodata == OUT_NODATA
        assert src.crs.to_epsg() == 4326
        assert src.is_tiled, "COG must be internally tiled"
        assert len(src.overviews(1)) > 0, "COG must have overviews"
        new = src.read(1)
        print(f"{name}: block {src.profile.get('blockxsize')} x {src.profile.get('blockysize')} | overviews {src.overviews(1)} | compression {src.compression}")

    with rasterio.open(source_tifs[name]) as src:
        old = src.read(1)
        old_nodata = src.nodata  # None for these files: empty cells are NaN

    # Valid pixels must be identical to the source (after the float64 -> float32 cast); only the nodata value was harmonised
    old_invalid = ~np.isfinite(old)
    if old_nodata is not None:
        old_invalid |= np.isclose(old, old_nodata)
    old = old.astype("float32")
    new_invalid = new == OUT_NODATA
    assert np.array_equal(old_invalid, new_invalid), "Valid-pixel mask changed"
    assert np.array_equal(old[~old_invalid], new[~new_invalid]), "Pixel values changed during COG conversion"
print("OK: soil sand COGs and metadata are ready for the STAC builder notebook.")

sand_5km_0-5cm: block 512 x 512 | overviews [2, 4, 8] | compression Compression.deflate
sand_5km_5-15cm: block 512 x 512 | overviews [2, 4, 8] | compression Compression.deflate
sand_5km_15-30cm: block 512 x 512 | overviews [2, 4, 8] | compression Compression.deflate
sand_5km_30-60cm: block 512 x 512 | overviews [2, 4, 8] | compression Compression.deflate
sand_5km_60-100cm: block 512 x 512 | overviews [2, 4, 8] | compression Compression.deflate
sand_5km_100-200cm: block 512 x 512 | overviews [2, 4, 8] | compression Compression.deflate
OK: soil sand COGs and metadata are ready for the STAC builder notebook.
